In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('raw-data/13F_ARK_Raw.csv')
df.head()

,Sym,Issuer Name,Cl,CUSIP,Value ($000),%,Shares,Principal,Option Type
0,TSLA,Tesla Inc,COMMON STOCK,88160R101,"1,052,546",8.2%,"2,831,329",NaN,NaN
1,AMD,Advanced Micro Devices Inc,COMMON STOCK,7903107,"551,875",4.3%,"2,712,854",NaN,NaN
2,CRSP,CRISPR Therapeutics AG,COMMON STOCK,H17182108,"538,189",4.2%,"11,313,623",NaN,NaN
3,SHOP,Shopify Inc,COMMON STOCK,82509L107,"495,553",3.9%,"4,177,652",NaN,NaN
4,PLTR,Palantir Technologies Inc,COMMON STOCK,69608A108,"454,913",3.5%,"3,109,881",NaN,NaN


In [3]:
print(df.columns.tolist())
df = df[['Sym',"Issuer Name", "Cl", '%']]
df.head()

['Sym', 'Issuer Name', 'Cl', 'CUSIP', 'Value ($000)', '%', 'Shares', 'Principal', 'Option Type']


,Sym,Issuer Name,Cl,%
0,TSLA,Tesla Inc,COMMON STOCK,8.2%
1,AMD,Advanced Micro Devices Inc,COMMON STOCK,4.3%
2,CRSP,CRISPR Therapeutics AG,COMMON STOCK,4.2%
3,SHOP,Shopify Inc,COMMON STOCK,3.9%
4,PLTR,Palantir Technologies Inc,COMMON STOCK,3.5%


In [4]:
def common_stock_filter(df):
    return df[df['Cl']== 'COMMON STOCK']
df_common = common_stock_filter(df)
df_common

,Sym,Issuer Name,Cl,%
0,TSLA,Tesla Inc,COMMON STOCK,8.2%
1,AMD,Advanced Micro Devices Inc,COMMON STOCK,4.3%
2,CRSP,CRISPR Therapeutics AG,COMMON STOCK,4.2%
3,SHOP,Shopify Inc,COMMON STOCK,3.9%
4,PLTR,Palantir Technologies Inc,COMMON STOCK,3.5%
...,...,...,...,...
173,KMT,Kennametal Inc,COMMON STOCK,0.0%
175,AVNT,Avient Corp,COMMON STOCK,0.0%
176,MTRN,Materion Corp,COMMON STOCK,0.0%
177,MMM,3M Co,COMMON STOCK,0.0%


In [5]:
import yfinance as yf
from pykrx import stock
from tqdm import tqdm

KRX 로그인 실패: KRX_ID 또는 KRX_PW 환경 변수가 설정되지 않았습니다.


In [6]:
def get_fundamentals(ticker):
    try:
        info = yf.Ticker(ticker).info
        return {
            'Sym':          ticker,
            'rev_growth':   info.get('revenueGrowth'),
            'gross_margin': info.get('grossMargins'),
            'ps_ratio':     info.get('priceToSalesTrailing12Months'),
            'market_cap':   info.get('marketCap'),
            'sector':       info.get('sector'),
            'per':          info.get('trailingPE'),
            'pbr':          info.get('priceToBook'),
        }
    except Exception as e:
        print(ticker, e)
        return None

records = [get_fundamentals(t) for t in tqdm(df_common['Sym'])]
fundamentals = pd.DataFrame([r for r in records if r])

ark_enriched = df_common.merge(fundamentals, on='Sym').dropna()

  0%|          | 0/158 [00:00<?, ?it/s]

  1%|          | 1/158 [00:00<02:28,  1.06it/s]

  1%|▏         | 2/158 [00:01<01:46,  1.47it/s]

  2%|▏         | 3/158 [00:01<01:28,  1.76it/s]

  3%|▎         | 4/158 [00:02<01:15,  2.03it/s]

...

 99%|█████████▊| 156/158 [01:06<00:00,  2.61it/s]

 99%|█████████▉| 157/158 [01:06<00:00,  2.64it/s]

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: AXL"}}}


100%|██████████| 158/158 [01:07<00:00,  1.83it/s]

100%|██████████| 158/158 [01:07<00:00,  2.34it/s]

In [7]:
ark_enriched.to_csv('raw-data/ark_enriched.csv', index=False)
print(f"Exported {len(ark_enriched)} rows with columns: {ark_enriched.columns.tolist()}")

Exported 88 rows with columns: ['Sym', 'Issuer Name', 'Cl', '%', 'rev_growth', 'gross_margin', 'ps_ratio', 'market_cap', 'sector', 'per', 'pbr']


In [8]:
ark_enriched

,Sym,Issuer Name,Cl,%,rev_growth,gross_margin,ps_ratio,market_cap,sector,per,pbr
0,TSLA,Tesla Inc,COMMON STOCK,8.2%,0.158,0.19065,16.138880,1.579657e+12,Consumer Cyclical,385.871550,19.208110
1,AMD,Advanced Micro Devices Inc,COMMON STOCK,4.3%,0.378,0.53060,25.290546,9.472321e+11,Technology,192.354300,14.689103
3,SHOP,Shopify Inc,COMMON STOCK,3.9%,0.343,0.47970,11.981741,1.481662e+11,Technology,111.941180,11.881374
4,PLTR,Palantir Technologies Inc,COMMON STOCK,3.5%,0.847,0.84074,53.538486,2.796944e+11,Technology,131.089890,33.097870
7,HOOD,Robinhood Markets Inc,COMMON STOCK,3.2%,0.151,0.92239,19.575687,9.030265e+10,Financial Services,48.679610,9.698259
...,...,...,...,...,...,...,...,...,...,...,...
152,JBL,Jabil Inc,COMMON STOCK,0.0%,0.118,0.09226,1.210752,4.066915e+10,Technology,48.305767,30.681313
153,KMT,Kennametal Inc,COMMON STOCK,0.0%,0.218,0.32078,1.250234,2.671164e+09,Industrials,19.691011,1.971649
154,AVNT,Avient Corp,COMMON STOCK,0.0%,0.025,0.32508,1.033102,3.389608e+09,Basic Materials,21.488370,1.408912
155,MTRN,Materion Corp,COMMON STOCK,0.0%,0.308,0.16403,3.228607,6.186153e+09,Basic Materials,81.254100,6.464297
